# L'encapsulation

L'encapsulation permet de **contrôler l'accès** aux attributs d'une classe. On distingue trois niveaux :

| Niveau | Préfixe | Accès | Exemple |
|---|---|---|---|
| **Public** | aucun | Partout | `self.nom` |
| **Protégé** | `_` | Simple **convention** : « réservé à la classe et ses sous-classes » — Python ne bloque rien | `self._nom` |
| **Privé** | `__` | Classe uniquement (via *name mangling*, contournable) | `self.__nom` |

> ⚠️ Python ne fait **pas de vraie encapsulation**. Il utilise des **conventions de nommage**. Par défaut, tous les attributs sont publics.
>
> Le préfixe `_` (protégé) en particulier n'est **qu'un accord entre développeurs** : c'est un panneau « privé, merci de ne pas toucher ». Techniquement, `objet._nom` reste accessible partout — Python n'interdit rien, il compte sur la discipline du lecteur.

### Le *name mangling* (privé)

Un attribut préfixé par `__` est renommé automatiquement par Python pour le rendre moins accessible :
- `self.__name` dans la classe `Chien` → devient `_Chien__name` à l'exécution

## Classe simple avec accès direct aux attributs

In [ ]:
class Rectangle():

    # Constructeur
    def __init__(self , l , h):
        # Attributs  
        self.largeur = l
        self.hauteur = h
    
        
    # Méthode de la classe Rectangle
    
    def Surface(self):
        return self.largeur * self.hauteur
    
    def __str__(self):
        return f"Rectangle de largeur {self.largeur} et de hauteur {self.hauteur} et de surface {self.Surface()}"

---

## Getters et Setters

Pour contrôler l'accès aux attributs, on définit des **getters** (lecture) et des **setters** (écriture) :

- **Getter** : méthode qui **renvoie** la valeur d'un attribut
- **Setter** : méthode qui **modifie** la valeur d'un attribut (avec validation possible)

In [ ]:
class Rectangle():

    def __init__(self, l, h):
        self._largeur = l    # préfixe _ : convention « ne pas toucher directement »
        self._hauteur = h

    # Getters : méthodes pour récupérer la largeur et la hauteur
    def get_hauteur(self):
        return self._hauteur

    def get_largeur(self):
        return self._largeur

    # Setters : méthodes pour changer la largeur et la hauteur (avec validation)
    def set_hauteur(self, h):
        if h > 0:
            self._hauteur = h

    def set_largeur(self, l):
        if l > 0:
            self._largeur = l


# Démonstration
r = Rectangle(3, 4)
print(r.get_largeur(), r.get_hauteur())   # 3 4

r.set_hauteur(10)                          # valeur valide → acceptée
print(r.get_hauteur())                     # 10

r.set_hauteur(-5)                          # valeur invalide → ignorée par le setter
print(r.get_hauteur())                     # toujours 10

---

## Le décorateur `@property`

Le décorateur `@property` permet de créer des getters/setters **élégants**, accessibles comme des attributs (sans parenthèses).

Il offre aussi `@attribut.setter` pour la modification et `@attribut.deleter` pour la suppression :

In [ ]:
class Rectangle():

    def __init__(self, l, h):
        # On passe par les setters : la validation s'applique dès la construction
        self.largeur = l
        self.hauteur = h

    @property
    def hauteur(self):
        return self.__hauteur

    @hauteur.setter
    def hauteur(self, h):
        if h > 0:
            self.__hauteur = h

    @hauteur.deleter
    def hauteur(self):
        del self.__hauteur

    @property
    def largeur(self):
        return self.__largeur

    @largeur.setter
    def largeur(self, l):
        if l > 0:
            self.__largeur = l

    @largeur.deleter
    def largeur(self):
        del self.__largeur


# Démonstration : les propriétés s'utilisent comme des attributs, sans parenthèses
r = Rectangle(3, 4)
print(r.largeur, r.hauteur)   # 3 4 — appelle les getters

r.hauteur = 10                # appelle le setter
print(r.hauteur)              # 10

del r.hauteur                 # appelle le deleter

In [ ]:
class Rectangle():

    # Constructeur : on passe par les setters pour valider dès la création
    def __init__(self, l, h):
        self.set_largeur(l)
        self.set_hauteur(h)

    def set_largeur(self, l):
        # Ignorer silencieusement une valeur invalide laisserait l'objet
        # sans attribut `largeur` → AttributeError plus tard, difficile à
        # comprendre. On préfère refuser proprement avec une exception :
        if l <= 0:
            raise ValueError(f"La largeur doit être strictement positive, reçu : {l}")
        self.largeur = l

    def set_hauteur(self, h):
        if h <= 0:
            raise ValueError(f"La hauteur doit être strictement positive, reçu : {h}")
        self.hauteur = h

    def surface(self):
        return self.largeur * self.hauteur

    def __str__(self):
        return f"Rectangle de largeur {self.largeur} et de hauteur {self.hauteur} et de surface {self.surface()}"


# Cas valide
r = Rectangle(3, 4)
print(r)

# Cas invalide : la création est refusée proprement, avec un message clair
try:
    mauvais = Rectangle(-2, 4)
except ValueError as e:
    print("Création refusée :", e)

---

## Le décorateur `@property` — référence complète

`@property` transforme une méthode en **attribut virtuel**. Il permet :
- D'accéder à une valeur calculée comme si c'était un attribut (`obj.aire` au lieu de `obj.aire()`)
- D'ajouter de la **validation** lors de la modification
- De rendre un attribut **lecture seule** (en ne définissant pas le setter)

| Décorateur | Rôle | Déclencheur |
|---|---|---|
| `@property` | Getter | `obj.attr` |
| `@attr.setter` | Setter | `obj.attr = valeur` |
| `@attr.deleter` | Deleter | `del obj.attr` |

In [ ]:
class Cercle:
    def __init__(self, rayon):
        self._rayon = rayon   # attribut privé (convention _)

    @property
    def rayon(self):
        """Getter : retourne le rayon."""
        return self._rayon

    @rayon.setter
    def rayon(self, valeur):
        """Setter : valide que le rayon est positif."""
        if valeur <= 0:
            raise ValueError(f"Le rayon doit être positif, reçu : {valeur}")
        self._rayon = valeur

    @rayon.deleter
    def rayon(self):
        """Deleter : réinitialise le rayon."""
        print("Suppression du rayon")
        del self._rayon

    @property
    def aire(self):
        """Propriété calculée (pas de setter → lecture seule)."""
        import math
        return math.pi * self._rayon ** 2


c = Cercle(5)
print(c.rayon)        # 5  — appelle le getter
print(c.aire)         # 78.53...  — propriété calculée

c.rayon = 10          # appelle le setter
# c.rayon = -1        # ValueError !

# c.aire = 100        # AttributeError — pas de setter défini

---

## Name mangling (attributs privés `__`)

Un attribut préfixé par `__` (double underscore) subit le **name mangling** : Python le renomme automatiquement en `_NomClasse__attribut`. Cela évite les collisions accidentelles lors de l'héritage.

> Ce n'est **pas** une vraie protection — l'attribut reste accessible de l'extérieur en utilisant le nom transformé. C'est une protection faible par convention.

In [ ]:
class CompteBancaire:
    def __init__(self, titulaire, solde):
        self.titulaire = titulaire    # public
        self._solde = solde           # protégé (convention)
        self.__pin = "1234"           # privé (name mangling)

    def verifier_pin(self, pin):
        return pin == self.__pin      # accès interne OK


compte = CompteBancaire("Alice", 1000)

print(compte.titulaire)       # Alice   — accès normal
print(compte._solde)          # 1000    — accessible, mais déconseillé
# print(compte.__pin)         # AttributeError
print(compte._CompteBancaire__pin)  # '1234' — accessible via le nom transformé

# Vérifier les attributs disponibles
print(vars(compte))           # {'titulaire': 'Alice', '_solde': 1000, '_CompteBancaire__pin': '1234'}